# ML Lab · Module 3, Unit 9: Forecasting inflation

**Prof. Dr. Mehmet Songur** · ML Lab, Module 3

Data: 13 monthly US series (FRED). The setup cell prepares a monthly table `G` (index: date). Columns: `inf` (annual inflation, %), `monthly` (monthly inflation, %), the annual change of 7 series (`CPILFESL`, `PPIACO`, `M2SL`, `WTISPLC`, `AHETPI`, `PAYEMS`, `INDPRO`), the level of 4 series (`UNRATE`, `FEDFUNDS`, `T10Y3MM`, `UMCSENT`) and `HOUST` (log). **Have Gemini write the code**, then **audit it**.

**Before you start:** to edit this notebook opened from GitHub, choose **File → Save a copy in Drive** and work in the copy that opens.

**Rules and order**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running. Gemini may add its code as a new cell below the comment; run the cells in order.
3. When the task code runs without errors, run the matching **Check** cell. If it reports a problem, fix it and rerun the task cell and the check.
4. If a check passes, it gives you a **completion code**; paste it and the task code into the box in ML Lab.

> **Data vintage:** the setup cell prints the download time and each series' last month. With live FRED data the numbers may differ slightly from the Lab's; to work on a fixed vintage use the `LAST_MONTH` setting in the setup cell (change only that line).
>
> **Retrospective experiment:** there is no CPI for October 2025 (the US federal government shutdown); don't fill missing months, leave out the rows that contain them. For simplicity we assume “this month's data are known at month end”; in reality CPI is published in the middle of the following month, and the other series have their own publication lags and revisions. A real-time forecast must use the data vintages known on the forecast date (ALFRED).

In [ ]:
#@title Setup — run it, do not change it

import io, base64, urllib.request, warnings, datetime
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, KFold
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("always", category=ConvergenceWarning)   # convergence warnings stay visible
LAST_MONTH = None   # For a fixed data vintage write a month (e.g. "2025-12-01"): later observations are dropped. None: latest data.

def fred_download(series_id):
    """Downloads a series from FRED; columns: date, <series_id>. FRED marks missing values with '.';
    they are deliberately turned into NaN and counted."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + series_id
    with urllib.request.urlopen(url) as response:
        df = pd.read_csv(io.BytesIO(response.read()))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    print(f"{series_id}: {len(df)} rows, {df[series_id].isna().sum()} missing, {df['date'].min():%Y-%m} – {df['date'].max():%Y-%m}")
    return df

codes = ['CPIAUCSL', 'CPILFESL', 'PPIACO', 'M2SL', 'WTISPLC', 'AHETPI', 'PAYEMS', 'INDPRO', 'UNRATE', 'FEDFUNDS', 'T10Y3MM', 'UMCSENT', 'HOUST']
print("Download time (UTC):", datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"))
raw = pd.concat([fred_download(k).set_index("date") for k in codes], axis=1).sort_index().asfreq("MS")
if LAST_MONTH is not None:
    raw = raw.loc[:pd.Timestamp(LAST_MONTH)]
    print("Fixed vintage: observations after", LAST_MONTH, "were dropped.")
G = pd.DataFrame(index=raw.index)
G["inf"] = raw["CPIAUCSL"].pct_change(12, fill_method=None) * 100     # annual inflation (%)
G["monthly"] = raw["CPIAUCSL"].pct_change(1, fill_method=None) * 100  # monthly inflation (%)
for k in ["CPILFESL", "PPIACO", "M2SL", "WTISPLC", "AHETPI", "PAYEMS", "INDPRO"]:
    G[k] = raw[k].pct_change(12, fill_method=None) * 100              # annual change (%)
for k in ["UNRATE", "FEDFUNDS", "T10Y3MM", "UMCSENT"]:
    G[k] = raw[k]                                                      # level
G["HOUST"] = np.log(raw["HOUST"])                                      # housing starts (log)
_G = G.copy()
START, CUT = pd.Timestamp("1984-01-01"), pd.Timestamp("2015-01-01")   # CUT: first test TARGET month
_REF = {}

def _k(s):
    return base64.b64decode(s).decode()

def _rm(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))

def _reference():
    if _REF:
        return _REF
    ft = {}
    for k in _G.columns:
        if k != "monthly":
            for g in range(3):
                ft[f"{k}_l{g}"] = _G[k].shift(g)
    for g in range(12):
        ft[f"monthly_l{g}"] = _G["monthly"].shift(g)
    X = pd.DataFrame(ft)
    t = pd.concat([X, _G["inf"].shift(-1).rename("target")], axis=1).loc[START:].dropna()
    t["target_month"] = t.index + pd.DateOffset(months=1)
    e, s = t[t["target_month"] < CUT], t[t["target_month"] >= CUT]
    c = list(X.columns)
    ols = make_pipeline(StandardScaler(), LinearRegression()).fit(e[c], e["target"])
    las = GridSearchCV(make_pipeline(StandardScaler(), Lasso(max_iter=50000)), {"lasso__alpha": np.logspace(-4, 0.5, 60)},
                       cv=TimeSeriesSplit(5), scoring="neg_mean_squared_error").fit(e[c], e["target"])
    _REF.update(n_test=len(s), n_train=len(e), naive=_rm(s["inf_l0"], s["target"]), ols=_rm(ols.predict(s[c]), s["target"]),
                lasso=_rm(las.predict(s[c]), s["target"]), last_row=t.index.max(), last_target=t["target_month"].max())
    return _REF

def check_A(table, train, test, rmse_naive, rmse_ols):
    if "target" not in table.columns:
        print("❌ No column called 'target' in 'table'."); return
    h = table["target"]
    if np.allclose(h.values, _G["inf"].reindex(h.index).values, equal_nan=True):
        print("❌ The target is not shifted: 'target' is this month's inflation. Next month's value needs shift(-1) (target leakage)."); return
    if not np.allclose(h.values, _G["inf"].shift(-1).reindex(h.index).values, equal_nan=True):
        print("❌ 'target' does not match next month's annual inflation (inf.shift(-1)). The target may be built differently or missing months may have been filled."); return
    if "target_month" not in table.columns:
        print("❌ No 'target_month' column. Add the month each row's target belongs to: table['target_month'] = table.index + pd.DateOffset(months=1)"); return
    if not (pd.to_datetime(table["target_month"]).values == (table.index + pd.DateOffset(months=1)).values).all():
        print("❌ 'target_month' should be one month after the feature month (the index)."); return
    if table.drop(columns=["target_month"]).isna().sum().sum() > 0:
        print("❌ The table has missing values; rows with missing months should be left out (dropna)."); return
    if len(table.columns) - 2 != 51:
        print(f"❌ 51 features expected, found {len(table.columns) - 2} (excluding target and target_month). Re-read the recipe: 13 series × 3 lags + 12 lags of monthly inflation."); return
    for name, d in [("train", train), ("test", test)]:
        if "target_month" not in d.columns or not d.index.is_monotonic_increasing:
            print(f"❌ '{name}' must be in date order and contain the 'target_month' column; no shuffling."); return
    if train["target_month"].max() >= CUT:
        print(f"❌ Training contains a row whose target is {train['target_month'].max():%Y-%m}. Split by TARGET month: the December 2014 row's target is January 2015 and belongs to the test."); return
    if test["target_month"].min() < CUT:
        print("❌ The test contains rows whose target is before 2015; test target months must be 2015-01 or later."); return
    r = _reference()
    if len(test) > r["n_test"]:
        print("❌ The test period has more months than expected: missing months may have been filled. Don't fill; leave them out."); return
    if len(test) < r["n_test"] or len(train) < r["n_train"]:
        print(f"❌ Fewer rows than expected (training {len(train)}/{r['n_train']}, test {len(test)}/{r['n_test']}). The cut-off or start date may differ."); return
    if train.index.min() > START + pd.Timedelta(days=40):
        print("❌ The training data do not start in 1984."); return
    if abs(float(rmse_naive) - r["naive"]) > 0.005:
        print("❌ The naive forecast's RMSE differs from the expected value. Is the forecast 'this month's annual inflation' (inf_l0), compared with the target?"); return
    if abs(float(rmse_ols) - r["ols"]) > 0.02:
        print("❌ OLS's test RMSE differs from the expected value. Were all 51 features used? Was the model built on training data only?"); return
    print(f"Last training target: {train['target_month'].max():%Y-%m} · first test target: {test['target_month'].min():%Y-%m} · test rows: {len(test)}")
    print(f"Data vintage: last complete feature row {r['last_row']:%Y-%m}, last target month {r['last_target']:%Y-%m}")
    print(f"✅ All checks passed (naive ≈ {r['naive']:.3f}, OLS ≈ {r['ols']:.3f}). Your completion code: " + _k("TUwzLUVORi00S1Fa"))

def check_B(search, rmse_lasso, selected, train, test):
    if not isinstance(search, GridSearchCV):
        print("❌ 'search' should be a GridSearchCV object. Feeding pre-scaled data to LassoCV carries each validation fold's own mean and deviation into the scaling; use Pipeline(StandardScaler, Lasso) + GridSearchCV."); return
    est = search.estimator
    if not (isinstance(est, Pipeline) and isinstance(est.steps[0][1], StandardScaler) and isinstance(est.steps[-1][1], Lasso)):
        print("❌ The searched model should be Pipeline(StandardScaler, Lasso): the scaler must be fitted in each fold on that fold's training part only."); return
    if isinstance(search.cv, KFold) or not isinstance(search.cv, TimeSeriesSplit):
        print("❌ λ was not chosen with time-respecting cross-validation. Use cv=TimeSeriesSplit(5); shuffled KFold shows the future."); return
    if search.scoring != "neg_mean_squared_error":
        print("❌ State the selection measure explicitly: scoring='neg_mean_squared_error' (MSE)."); return
    if not hasattr(search, "best_estimator_"):
        print("❌ The search is not finished or refit=False: the model must be refitted on all training data with the chosen λ."); return
    final = search.best_estimator_
    names = list(getattr(final, "feature_names_in_", []))
    if len(names) != 51:
        print("❌ The model should be fitted with the 51 feature names (train[features])."); return
    if train["target_month"].max() >= CUT or test["target_month"].min() < CUT:
        print("❌ 'train' and 'test' are not split by target month (Task A)."); return
    if final[0].n_samples_seen_ != len(train) or not np.allclose(final[0].mean_, train[names].mean().values):
        print("❌ The final model was not fitted on the 'train' rows only: it should be fit(train[features], train['target'])."); return
    if abs(_rm(search.predict(test[names]), test["target"]) - float(rmse_lasso)) > 1e-6:
        print("❌ 'rmse_lasso' should be computed between search.predict(test[features]) and test['target'] (predictions on the test rows)."); return
    r = _reference()
    if abs(float(rmse_lasso) - r["lasso"]) > 0.01:
        print("❌ Lasso's test RMSE differs from the expected value. Do the λ grid ({'lasso__alpha': np.logspace(-4, 0.5, 60)}) and max_iter=50000 match the prompt?"); return
    nz = int(np.sum(np.abs(final[-1].coef_) > 0))
    if len(selected) != nz:
        print(f"❌ 'selected' has {len(selected)} items, but there are {nz} non-zero coefficients. Count negative coefficients too (coef_ != 0)."); return
    if "monthly_l11" not in selected:
        print("ℹ️ Note: monthly_l11 (base effect) is not on the list; check the feature names.")
    print(f"Chosen λ: {search.best_params_['lasso__alpha']:.4f} · selected variables: {nz} / 51 · test rows: {len(test)}")
    print(f"✅ All checks passed (Lasso ≈ {r['lasso']:.3f}, naive ≈ {r['naive']:.3f}). Your completion code: " + _k("TUwzLUxTTy03WFBE"))

print("G table:", G.shape)
G.tail(3).round(2)

## Task A — Problem and benchmark

1. **Features:** for every column of `G` except `monthly`, its values lagged 0, 1 and 2 months (`{column}_l0`, `{column}_l1`, `{column}_l2`), and `monthly` lagged 0–11 months (`monthly_l0` … `monthly_l11`): 51 in total.
2. **Target:** next month's annual inflation, a column called `target`: `G["inf"].shift(-1)`.
3. Combine the features and the target in a table called `table`, start it at `1984-01-01` and drop rows with missing values. Then add the month the target belongs to: `table["target_month"] = table.index + pd.DateOffset(months=1)` (the index is the feature month).
4. **Split by target month:** rows whose `target_month` is before `pd.Timestamp("2015-01-01")` → `train`, the rest → `test`. Put the feature names in a `features` list. Print the last training target and the first test target.
5. Over the test period, compute the RMSE of the **naive forecast** (`inf_l0`) and of a **51-variable OLS** standardised and fitted on the training data: `rmse_naive`, `rmse_ols`.

**Audit questions:** Is the target `shift(-1)`? Any negative shifts in the features? Is the cut-off by target month (the December 2014 row's target is January 2015)? Is a `pd.Timedelta` subtracted from a date string (`TypeError`; use `pd.Timestamp` first)? Any shuffling in the split? Were missing months filled?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(table, train, test, rmse_naive, rmse_ols)

## Task B — Lasso

1. Put `StandardScaler` and `Lasso(max_iter=50000)` in a `Pipeline` with `make_pipeline`: in each validation fold the scaling should be learned from that fold's training part only.
2. Choose λ with `GridSearchCV(pipe, {"lasso__alpha": np.logspace(-4, 0.5, 60)}, cv=TimeSeriesSplit(5), scoring="neg_mean_squared_error")`; store the search object in `search` and `fit` it on `train[features]`, `train["target"]` (after the search it is refitted on all training data with the chosen λ).
3. Compute the test RMSE with `search.predict(test[features])`, store it in `rmse_lasso` and compare it with the naive forecast.
4. Put the final model's (`search.best_estimator_[-1]`) features with non-zero coefficients in a `selected` list; print the top 8 by absolute value with their coefficients.

**Audit questions:** Is the scaler inside the folds? What is `cv`? Is `scoring` explicitly MSE? What do the selected variables say economically (base effect: `monthly_l11`)?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(search, rmse_lasso, selected, train, test)

## Your audit notes

- How did Gemini build the target and the split? Did you need to change it?
- How were scaling and cross-validation done?
- By how much does Lasso beat the naive forecast? With which data vintage (download time, last target month)? Which variables were selected, and what do they mean?
- What did the checks not test? (e.g. release calendars, data revisions)